# 02 — Análise exploratória

> **Onde está a variação evitável nas internações do SUS — em tempo de permanência, custo e
> mortalidade hospitalar — e quais fatores a explicam?**

As cinco sub-perguntas do [README](../README.md), uma seção cada. Algumas análises estão
resolvidas como ponto de partida; as marcadas **Sua vez** são suas.

Regra de ouro desta fase: **um gráfico, uma mensagem.** Se você não consegue escrever em uma frase
o que o gráfico mostra, o gráfico ainda não está pronto.

## Preparação

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = Path.cwd().parents[2]
sys.path.insert(0, str(Path.cwd().parent / "src"))

import dicionario as dic
import visual as vis

vis.aplicar_estilo()
pd.set_option("display.width", 150)

UF, ANO = "SC", 2024
caminho = RAIZ / "dados" / "bruto" / "sihsus" / f"analitico-{UF}-{ANO}.parquet"
if not caminho.exists():
    caminho = RAIZ / "dados" / "exemplo" / f"sihsus-{UF}-{ANO}-amostra.parquet"
    print("Parquet completo não encontrado; usando a amostra de exemplo.")

df = pd.read_parquet(caminho)
print(f"{len(df):,} internações".replace(",", ".") + f" | {caminho.name}")

## Sub-pergunta 1 — Qual é o perfil das internações?

Antes de qualquer cruzamento: quem interna, por quê, e em que caráter.

### 1.1 Distribuição por faixa de idade e sexo

In [ ]:
perfil = (df.groupby(["faixa_idade", "sexo"], observed=True).size()
            .unstack("sexo")
            .reindex(dic.ROTULOS_IDADE)
            .fillna(0))

fig, ax = plt.subplots(figsize=(9, 5))
y = np.arange(len(perfil))
altura = 0.40   # 0.40 com deslocamento de 0.20 deixa uma folga entre as barras do par
ax.barh(y + 0.20, perfil["Feminino"], height=altura * 0.9,
        color=vis.CATEGORICA[0], label="Feminino")
ax.barh(y - 0.20, perfil["Masculino"], height=altura * 0.9,
        color=vis.CATEGORICA[1], label="Masculino")

ax.set_yticks(y, perfil.index)
ax.set_xlabel("internações")
vis.eixo_milhares(ax)
ax.grid(False); ax.xaxis.grid(True)
ax.invert_yaxis()
vis.titular(ax, "Internações por faixa de idade e sexo",
            f"{UF}, {ANO} — {len(df):,} AIH".replace(",", "."))
ax.legend(loc="lower right")
plt.show()

A leitura esperada: um pico em 15–29 anos puxado por obstetrícia, e um segundo movimento a partir
dos 60. Confirme isso na próxima célula antes de aceitar a hipótese — é fácil ver o que se espera
ver.

In [ ]:
jovens = df[df["faixa_idade"] == "15 a 29"]
print("principais capítulos CID-10 na faixa de 15 a 29 anos:")
print((jovens["cid_grupo"].value_counts(normalize=True).head(5) * 100)
      .round(1).astype(str).add(" %").to_string())

### 1.2 Principais causas de internação

In [ ]:
causas = df["cid_grupo"].value_counts().head(10).sort_values()

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.barh(causas.index, causas.values, color=vis.CATEGORICA[0], height=0.68)
ax.set_xlabel("internações")
ax.grid(False); ax.xaxis.grid(True)
ax.set_xlim(0, causas.max() * 1.12)
vis.eixo_milhares(ax)
vis.rotular_barras(ax, horizontal=True)
vis.titular(ax, "Dez principais capítulos CID-10 por número de internações",
            f"{UF}, {ANO}")
plt.show()

**Sua vez.** Responda, com número e gráfico:

1. Como se distribuem as internações entre **eletiva e urgência**, e isso muda por faixa de idade?
2. Qual a participação de cada **especialidade de leito**?
3. Qual proporção usou **UTI**, e como isso varia por complexidade?

In [ ]:
# sua análise

## Sub-pergunta 2 — O que explica a variação no tempo de permanência?

`DIAS_PERM` é fortemente assimétrico: a maioria das internações é curta e uma cauda longa puxa a
média. **Use a mediana**, não a média — e diga por quê no relatório.

In [ ]:
print(df["DIAS_PERM"].describe().to_string())
print(f"\nmédia   : {df['DIAS_PERM'].mean():.1f} dias")
print(f"mediana : {df['DIAS_PERM'].median():.0f} dias")
print(f"p95     : {df['DIAS_PERM'].quantile(0.95):.0f} dias")
print(f"máximo  : {df['DIAS_PERM'].max()} dias")
print(f"\ninternações de 0 dia (entrada e saída no mesmo dia): "
      f"{(df['DIAS_PERM'] == 0).mean():.1%}")

**Sua vez.**

1. Desenhe a distribuição de `DIAS_PERM`. Um histograma comum vai ficar ilegível por causa da
   cauda — resolva isso (corte no p99, ou escala logarítmica no eixo de contagem) e **diga no
   subtítulo o que você cortou**.
2. Compare a **mediana** de permanência por especialidade, por complexidade e por uso de UTI.
   Qual fator separa mais?
3. A permanência mediana muda ao longo do ano? Cuidado com as bordas de competência.

In [ ]:
# sua análise

## Sub-pergunta 3 — Como o gasto se concentra?

Aqui cabe um aviso de desenho: a curva de Pareto costuma ser desenhada com barras de valor e uma
linha de percentual acumulado em um **segundo eixo y**. Não faça isso — dois eixos y com escalas
diferentes é o erro de gráfico mais comum que existe, porque o cruzamento das duas séries passa a
depender da escala escolhida, e não do dado. Se a pergunta é concentração, desenhe **só a
participação acumulada**, em um eixo só.

In [ ]:
gasto = df.groupby("cid_grupo", observed=True)["VAL_TOT"].sum().sort_values(ascending=False)
acumulado = (gasto.cumsum() / gasto.sum() * 100)
posicao = np.arange(1, len(acumulado) + 1)
ate_80 = int((acumulado < 80).sum() + 1)

fig, ax = plt.subplots(figsize=(9, 5))
ax.step(posicao, acumulado.values, where="post", color=vis.CATEGORICA[0])
ax.plot(posicao, acumulado.values, "o", color=vis.CATEGORICA[0], markersize=4)
ax.axhline(80, color=vis.TINTA_FRACA, linewidth=1, linestyle=(0, (4, 3)))
ax.text(len(posicao), 80.8, "80 % do gasto", ha="right", va="bottom",
        fontsize=9, color=vis.TINTA_SECUNDARIA)
ax.annotate(f"{ate_80} capítulos", xy=(ate_80, acumulado.iloc[ate_80 - 1]),
            xytext=(ate_80 + 1.2, 58), fontsize=9.5, color=vis.TINTA,
            arrowprops=dict(arrowstyle="-", color=vis.TINTA_FRACA, linewidth=1))

ax.set_xlabel("capítulos CID-10, ordenados por gasto")
ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))  # o eixo conta capítulos, não frações
ax.set_ylabel("participação acumulada (%)")
ax.set_ylim(0, 102); ax.set_xlim(0.5, len(posicao) + 0.5)
vis.titular(ax, "O gasto se concentra em poucos capítulos do CID-10",
            f"{UF}, {ANO} — R$ {gasto.sum()/1e6:,.1f} milhões no total".replace(",", "."))
plt.show()

print(f"{ate_80} de {len(gasto)} capítulos concentram 80% do gasto.")
print("\ntop 5 por gasto:")
print((gasto.head(5) / 1e6).round(1).astype(str).add(" mi").to_string())

**Sua vez.**

1. Refaça a concentração por **procedimento** (`PROC_REA`) e por **estabelecimento** (`CNES`).
   A concentração é maior ou menor que por capítulo CID? O que isso sugere?
2. Separe **valor total** de **valor médio por internação**. Um capítulo pode pesar por volume
   (muitas internações baratas) ou por complexidade (poucas caras) — e a ação de gestão é
   diferente em cada caso.
3. Quanto do gasto vai para internações que usaram UTI?

In [ ]:
# sua análise

## Sub-pergunta 4 — O que está associado à mortalidade hospitalar?

Atenção à linguagem: o que segue é **associação**, não causa. A idade não "causa" o óbito da
internação; ela é um marcador de gravidade, e a base não tem as variáveis clínicas que explicariam
o mecanismo.

In [ ]:
mortalidade = (df.groupby("faixa_idade", observed=True)
                 .agg(internacoes=("MORTE", "size"), obitos=("MORTE", "sum"))
                 .reindex(dic.ROTULOS_IDADE)
                 .dropna())
mortalidade["taxa"] = mortalidade["obitos"] / mortalidade["internacoes"] * 100

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(mortalidade.index, mortalidade["taxa"], color=vis.CATEGORICA[0], width=0.62)
ax.set_ylabel("óbitos por 100 internações")
ax.set_ylim(0, mortalidade["taxa"].max() * 1.18)
for x, (taxa, n_int) in enumerate(zip(mortalidade["taxa"], mortalidade["internacoes"])):
    ax.text(x, taxa + mortalidade["taxa"].max() * 0.03, f"{taxa:.1f}%",
            ha="center", fontsize=9, color=vis.TINTA_SECUNDARIA)
vis.titular(ax, "Mortalidade hospitalar cresce com a idade",
            f"{UF}, {ANO} — taxa geral de {df['MORTE'].mean():.2%}; "
            "o rótulo mostra a taxa, não o número de óbitos")
plt.xticks(rotation=20, ha="right")
plt.show()

print(mortalidade.to_string())

**Sua vez.**

1. Compare a mortalidade entre **eletiva e urgência**, e entre **com e sem UTI**. Antes de
   interpretar, pense: o que a seleção de casos faz com esses números?
2. Quais capítulos CID-10 têm a maior letalidade hospitalar? Exclua os de baixo volume — uma taxa
   sobre 12 internações não significa nada. Defina e **declare** o corte mínimo que você usou.
3. Escreva um parágrafo sobre por que comparar mortalidade bruta entre hospitais é enganoso.

In [ ]:
# sua análise

## Sub-pergunta 5 — O que o deslocamento entre municípios revela?

`MUNIC_RES ≠ MUNIC_MOV` significa que o paciente foi atendido fora do município onde mora. Isso
mistura dois fenômenos — busca por serviço que não existe na origem, e erro de cadastro — e essa
ambiguidade precisa aparecer na conclusão.

In [ ]:
print(f"internações fora do município de residência: {df['fora_do_municipio'].mean():.1%}")
print("\npor complexidade:")
print((df.groupby("complexidade", observed=True)["fora_do_municipio"].mean() * 100)
      .round(1).astype(str).add(" %").to_string())

**Sua vez.**

1. Quais municípios são **origem** líquida e quais são **destino** líquido de internações?
2. O deslocamento é maior na alta complexidade? Isso é esperado — mas confirme com número.
3. Quem se desloca tem permanência ou custo diferente? Cuidado: pacientes transferidos tendem a
   ser mais graves, então a comparação bruta confunde deslocamento com gravidade.

In [ ]:
# sua análise

## Fechamento — o que vai para o relatório

`relatorio.md` é escrito para alguém que **não vai abrir este notebook**. Estrutura:

1. **Pergunta** — e por que ela importa para uma decisão de gestão.
2. **Dados** — fonte, escopo, e o que foi descartado na preparação.
3. **Achados** — um por sub-pergunta, cada um com o número que o sustenta e o gráfico que o mostra.
4. **Limitações** — as advertências de domínio que sobreviveram, com a consequência prática de cada
   uma. Esta seção é o que separa análise de opinião.

Três frases que **não** podem aparecer no relatório, e o que colocar no lugar:

| Não escreva | Escreva |
|---|---|
| "N pacientes internaram" | "N internações" — não há identificador de paciente no SIH |
| "a idade causa maior mortalidade" | "a mortalidade é maior nas faixas mais idosas" |
| "o hospital X é pior" | "o hospital X tem mortalidade bruta maior, sem ajuste por gravidade" |